# Superstore PySpark — Tier 1 Advanced Topics

This notebook covers the advanced Tier 1 PySpark topics applied to the Superstore dataset.

**Topics covered:**
- UDFs (User Defined Functions) — regular and Pandas UDF
- Null handling — check, dropna, fillna, isNull/isNotNull
- Schema enforcement with StructType
- Repartition vs Coalesce
- Partition count
- Writing output to Parquet and CSV

**Dataset:** Sample Superstore (9,994 records)  
**Stack:** PySpark, Python 3.13, Java 17

## 1. Environment Setup & SparkSession

In [ ]:
import os

# Force Spark to bind to localhost — avoids IP binding issues on laptops
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["JAVA_HOME"]      = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"]           = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ["PATH"]

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Superstore_Tier1_Advanced")
    .config("spark.driver.host",        "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")
print("SparkSession ready.")

## 2. Load Data

In [ ]:
from pyspark.sql.functions import col

df = (
    spark.read
    .option("header",    True)
    .option("inferSchema", True)
    .option("quote",     '"')
    .option("escape",    '"')
    .option("multiLine", True)
    .csv("data/Sample - Superstore.csv")
)

# Cast numeric columns explicitly — inferSchema can mis-infer these
df = df \
    .withColumn("Sales",    col("Sales").cast("double")) \
    .withColumn("Profit",   col("Profit").cast("double")) \
    .withColumn("Discount", col("Discount").cast("double")) \
    .withColumn("Quantity", col("Quantity").cast("integer"))

print(f"Total records: {df.count()}")
df.printSchema()

## 3. UDFs — User Defined Functions

UDFs allow custom Python logic to run inside Spark transformations.

**Key rule:** Always prefer native Spark functions (`when`, `col`, `F.sum`) over UDFs.
Native functions run inside the JVM and are fully optimized.
UDFs run in Python and break JVM optimization — use only when native functions can't express the logic.

**Always null-guard UDFs** — Spark passes `None` for null values and unguarded UDFs will crash.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, IntegerType

# UDF 1 — Discount Risk
def discount_risk(x):
    if x is None: return "Unknown"    # always null-guard
    if x >= 0.30:   return "High Risk"
    elif x >= 0.15: return "Medium Risk"
    else:           return "Safe"

discount_risk_udf = udf(discount_risk, StringType())

# UDF 2 — Profit Category
def profit_category(profit, sales):
    if sales is None or sales == 0: return "Unknown"
    margin = (profit / sales) * 100
    if margin > 20:   return "High Profit"
    elif margin >= 0: return "Break Even"
    else:             return "Loss"

profit_category_udf = udf(profit_category, StringType())

# UDF 3 — Product Score
def product_score(profit, sales, quantity, discount):
    if any(v is None for v in [profit, sales, quantity, discount]):
        return 0
    score = 0
    if profit   > 200:  score += 20
    if sales    > 1000: score += 30
    if quantity > 5:    score += 25
    if discount < 0.15: score += 25
    return score

product_score_udf = udf(product_score, IntegerType())

print("UDFs registered successfully.")

In [ ]:
# Apply UDFs to DataFrame
df_enriched = df \
    .withColumn("Discount_Risk",   discount_risk_udf(col("Discount"))) \
    .withColumn("Profit_Category", profit_category_udf(col("Profit"), col("Sales"))) \
    .withColumn("Product_Score",   product_score_udf(
        col("Profit"), col("Sales"), col("Quantity"), col("Discount")
    ))

df_enriched.select(
    "Category", "Sales", "Profit", "Discount",
    "Discount_Risk", "Profit_Category", "Product_Score"
).show(10)

In [ ]:
# Compare UDF vs native Spark functions — same result, different performance
from pyspark.sql.functions import when

df_native = df.withColumn(
    "Discount_Risk_Native",
    when(col("Discount") >= 0.30, "High Risk")
    .when(col("Discount") >= 0.15, "Medium Risk")
    .otherwise("Safe")
).withColumn(
    "Discount_Risk_UDF", discount_risk_udf(col("Discount"))
)

# Both columns should be identical
df_native.select("Discount", "Discount_Risk_Native", "Discount_Risk_UDF").show(10)

# Key interview point:
# native when() → runs inside JVM → fast
# UDF           → runs in Python  → slower due to serialization overhead

In [ ]:
# Pandas UDF — faster than regular UDF (uses Apache Arrow for batch processing)
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf(StringType())
def discount_risk_pandas_udf(discount: pd.Series) -> pd.Series:
    def classify(x):
        if x >= 0.30:   return "High Risk"
        elif x >= 0.15: return "Medium Risk"
        else:           return "Safe"
    return discount.apply(classify)

df.withColumn(
    "Discount_Risk_Pandas", discount_risk_pandas_udf(col("Discount"))
).select("Discount", "Discount_Risk_Pandas").show(10)

# Performance order: native when() > Pandas UDF > regular UDF

## 4. Null Handling

Real production data always has nulls. Superstore is clean (0 nulls) so we create
a test DataFrame with nulls to practice all null-handling patterns.

In [ ]:
import pyspark.sql.functions as F

# Check nulls across all columns in the real Superstore dataset
df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

print(f"Total records: {df.count()}")

In [ ]:
from pyspark.sql import Row

# Create test DataFrame with intentional nulls for practice
test_df = spark.createDataFrame([
    Row(Category="Furniture",       Sales=None,   Profit=100.0, Discount=0.2),
    Row(Category=None,              Sales=500.0,  Profit=None,  Discount=0.1),
    Row(Category="Technology",      Sales=1000.0, Profit=200.0, Discount=None),
    Row(Category="Office Supplies", Sales=300.0,  Profit=50.0,  Discount=0.05),
    Row(Category=None,              Sales=None,   Profit=None,  Discount=None),
])

print("Test DataFrame with nulls:")
test_df.show()

In [ ]:
# dropna — different strategies

# Drop rows where ANY column is null
print("dropna() — drop if ANY null:")
test_df.dropna().show()

# Drop rows where ALL columns are null
print("dropna(how='all') — drop if ALL null:")
test_df.dropna(how="all").show()

# Drop rows where specific columns are null
print("dropna(subset=['Category', 'Sales']) — drop if these cols null:")
test_df.dropna(subset=["Category", "Sales"]).show()

# Keep rows with at least N non-null values
print("dropna(thresh=3) — keep if at least 3 non-null values:")
test_df.dropna(thresh=3).show()

In [ ]:
# fillna — fill nulls with specific values per column
print("fillna with specific values per column:")
test_df.fillna({
    "Category": "Unknown",
    "Sales":    0.0,
    "Profit":   0.0,
    "Discount": 0.0
}).show()

In [ ]:
# isNull / isNotNull — filtering

print("Rows where Sales is null:")
test_df.filter(F.col("Sales").isNull()).show()

print("Rows where Sales is NOT null:")
test_df.filter(F.col("Sales").isNotNull()).show()

print("Rows where both Sales and Category are NOT null:")
test_df.filter(
    F.col("Sales").isNotNull() & F.col("Category").isNotNull()
).show()

In [ ]:
# UDF null guard demonstration — critical production pattern

# Without null guard — crashes on null input
def discount_risk_unsafe(x):
    if x >= 0.30:   return "High Risk"   # crashes if x is None
    elif x >= 0.15: return "Medium Risk"
    else:           return "Safe"

discount_risk_unsafe_udf = udf(discount_risk_unsafe, StringType())

# Safe version — always guard None in UDFs
def discount_risk_safe(x):
    if x is None: return "Unknown"       # null guard
    if x >= 0.30:   return "High Risk"
    elif x >= 0.15: return "Medium Risk"
    else:           return "Safe"

discount_risk_safe_udf = udf(discount_risk_safe, StringType())

print("Safe UDF on data with nulls:")
test_df.withColumn(
    "Discount_Risk", discount_risk_safe_udf(F.col("Discount"))
).show()

In [ ]:
# Closing exercise — apply fillna + enrichment to real Superstore df
df_clean = df.fillna({
    "Sales":    0.0,
    "Profit":   0.0,
    "Discount": 0.0,
    "Quantity": 0
})

# Verify zero nulls in numeric columns
df_clean.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in ["Sales", "Profit", "Discount", "Quantity"]
]).show()

# Apply UDF enrichment on clean data
df_final = df_clean \
    .withColumn("Discount_Risk",   discount_risk_safe_udf(col("Discount"))) \
    .withColumn("Profit_Category", profit_category_udf(col("Profit"), col("Sales"))) \
    .withColumn("Product_Score",   product_score_udf(
        col("Profit"), col("Sales"), col("Quantity"), col("Discount")
    ))

df_final.select(
    "Category", "Sales", "Discount_Risk", "Profit_Category", "Product_Score"
).show(10)

print(f"Final record count: {df_final.count()}")

## 5. Schema Enforcement with StructType

Defining schema explicitly instead of relying on `inferSchema`.

**Why explicit schema?**
- Faster — no double file scan
- Deterministic — no type-inference surprises
- Safer in production — avoids runtime failures from mis-inferred types

This is the recommended approach for production pipelines.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType
)

# Define full Superstore schema explicitly
schema = StructType([
    StructField("Row ID",        IntegerType(), True),
    StructField("Order ID",      StringType(),  True),
    StructField("Order Date",    StringType(),  True),
    StructField("Ship Date",     StringType(),  True),
    StructField("Ship Mode",     StringType(),  True),
    StructField("Customer ID",   StringType(),  True),
    StructField("Customer Name", StringType(),  True),
    StructField("Segment",       StringType(),  True),
    StructField("Country",       StringType(),  True),
    StructField("City",          StringType(),  True),
    StructField("State",         StringType(),  True),
    StructField("Postal Code",   StringType(),  True),
    StructField("Region",        StringType(),  True),
    StructField("Product ID",    StringType(),  True),
    StructField("Category",      StringType(),  True),
    StructField("Sub-Category",  StringType(),  True),
    StructField("Product Name",  StringType(),  True),
    StructField("Sales",         DoubleType(),  True),
    StructField("Quantity",      IntegerType(), True),
    StructField("Discount",      DoubleType(),  True),
    StructField("Profit",        DoubleType(),  True),
])

# Read with explicit schema — no inferSchema needed
df_schema = (
    spark.read
    .option("header",    True)
    .option("quote",     '"')
    .option("escape",    '"')
    .option("multiLine", True)
    .schema(schema)
    .csv("data/Sample - Superstore.csv")
)

# Verify schema — Sales should show as double, not string
df_schema.printSchema()
df_schema.select("Sales", "Profit", "Discount", "Quantity").show(5)
print(f"Total records: {df_schema.count()}")

## 6. Repartition vs Coalesce

Controls how data is distributed across partitions (parallel processing units).

| | `repartition` | `coalesce` |
|---|---|---|
| Direction | Increase or decrease | Decrease only |
| Shuffle | Yes (full shuffle) | No shuffle |
| Speed | Slower | Faster |
| Use for | Increasing partitions, join optimization | Reducing partitions, single CSV write |

In [ ]:
# Check current partition count
print(f"Current partitions: {df_schema.rdd.getNumPartitions()}")

In [ ]:
# repartition — full shuffle, can increase or decrease
df_rep4 = df_schema.repartition(4)
print(f"After repartition(4): {df_rep4.rdd.getNumPartitions()} partitions")

# repartition by column — co-locates same Category on same partition
# useful before groupBy or join on that column
df_rep_cat = df_schema.repartition(4, "Category")
print(f"After repartition(4, 'Category'): {df_rep_cat.rdd.getNumPartitions()} partitions")

In [ ]:
# coalesce — no shuffle, only reduces partitions
df_coal = df_schema.coalesce(2)
print(f"After coalesce(2): {df_coal.rdd.getNumPartitions()} partitions")

# coalesce(1) — merge into single partition for single file CSV output
df_coal1 = df_schema.coalesce(1)
print(f"After coalesce(1): {df_coal1.rdd.getNumPartitions()} partition")

## 7. Writing Output

Writing the final enriched DataFrame to Parquet and CSV.

- **Parquet** — standard for data lakes, columnar, compressed, fast
- **CSV** — for compatibility, use `coalesce(1)` for single file output
- Always specify `header=True` on **both** write and read independently

In [ ]:
import os
os.makedirs("output", exist_ok=True)

# Write enriched DataFrame as Parquet
df_final.write.mode("overwrite").parquet("output/superstore_enriched_parquet")
print("Written: output/superstore_enriched_parquet")

# Write schema-enforced DataFrame as CSV — single file
df_schema.coalesce(1) \
    .write.mode("overwrite") \
    .option("header", True) \
    .csv("output/superstore_schema_csv")
print("Written: output/superstore_schema_csv")

In [ ]:
# Verify read-back — must specify header=True on read too
print("Parquet verification:")
spark.read.parquet("output/superstore_enriched_parquet") \
    .select("Category", "Sales", "Discount_Risk", "Profit_Category", "Product_Score") \
    .show(5)

print("CSV verification:")
spark.read \
    .option("header", True) \
    .csv("output/superstore_schema_csv") \
    .show(5)

## Summary

**Topics closed in this notebook:**

| Topic | Key Takeaway |
|---|---|
| UDFs | Always null-guard; prefer native `when()` over UDFs; Pandas UDF faster than regular UDF |
| Null handling | `dropna` (any/all/subset/thresh), `fillna` per column, `isNull`/`isNotNull` filters |
| Schema enforcement | Explicit schema = faster, deterministic, safer than `inferSchema` |
| Repartition | Full shuffle, can increase or decrease, use for join optimization |
| Coalesce | No shuffle, decreases only, use for reducing partitions or single CSV write |
| Output | Parquet for data lakes, CSV with `coalesce(1)` for single file; `header=True` on both read and write |

**PySpark Tier 1 — Complete ✅**

**Next — Tier 2:**
- Caching and persistence (`df.cache()`, `df.persist()`)
- Spark UI — reading job DAGs and identifying slow stages
- Delta Lake basics

**Repo:** https://github.com/surabhiks2k/superstore-data-platform